In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
from torch.utils.data import Dataset, DataLoader
import os

# ── Config ────────────────────────────────────────────────
LOOKBACK    = 100
HORIZON     = 30
BATCH       = 32
EPOCHS      = 50
LR          = 1e-3
PATIENCE    = 10
D_MODEL     = 64
N_HEADS     = 4
N_LAYERS    = 3
D_FF        = 128
DROPOUT     = 0.1

# Multi-feature input — this is the key change from before
FEATURE_COLS = ['avg_speed', 'vehicle_count', 'total_flow', 'flow_imbalance']
INPUT_DIM    = len(FEATURE_COLS)  # 4, not 1

TARGET_COL   = 'avg_speed'       # predict speed, not congestion score

CYCLICAL_SOURCES = [
    'B.Dongying Road2',
    "E.People's Hall1",
    "E.People's Hall2",
    "D.Xing'an South Road1",
    "D.Xing'an South Road2",
    'G.Xinhua Square1',
    'G.Xinhua Square2',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Input features: {FEATURE_COLS}")
print(f"Input dimension: {INPUT_DIM}")

Device: cpu
Input features: ['avg_speed', 'vehicle_count', 'total_flow', 'flow_imbalance']
Input dimension: 4


In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(1))

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])

In [3]:
class MultiFeatureTransformer(nn.Module):
    """
    Transformer encoder for traffic forecasting.
    Takes multiple features per timestep as input.
    Exposes get_embedding() for GNN node feature extraction.
    """
    def __init__(self, input_dim=4, d_model=64, nhead=4,
                 num_encoder_layers=3, dim_feedforward=128,
                 dropout=0.1, lookback=100, horizon=30):
        super().__init__()
        self.input_dim = input_dim
        self.d_model   = d_model
        self.lookback  = lookback
        self.horizon   = horizon

        # Project input_dim features → d_model
        # This is the only change from your original model
        self.input_proj = nn.Linear(input_dim, d_model)

        self.pos_enc = PositionalEncoding(
            d_model, max_len=lookback + 10, dropout=dropout
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers
        )

        self.decoder = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, horizon)
        )

    def get_embedding(self, x):
        """
        Extract embedding vector before decoder.
        x: (batch, lookback, input_dim)
        returns: (batch, d_model)
        """
        x = x.permute(1, 0, 2)           # (lookback, batch, input_dim)
        x = self.input_proj(x)            # (lookback, batch, d_model)
        x = self.pos_enc(x)               # (lookback, batch, d_model)
        x = self.transformer_encoder(x)  # (lookback, batch, d_model)
        return x[-1]                      # (batch, d_model)

    def forward(self, x):
        emb = self.get_embedding(x)
        return self.decoder(emb)          # (batch, horizon)


model = MultiFeatureTransformer(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    nhead=N_HEADS,
    num_encoder_layers=N_LAYERS,
    dim_feedforward=D_FF,
    dropout=DROPOUT,
    lookback=LOOKBACK,
    horizon=HORIZON
).to(device)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total:,}")

Trainable parameters: 112,926


C:\Users\Jaya\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [4]:
class MultiFeatureDataset(Dataset):
    def __init__(self, X, y):
        # X shape: (samples, lookback, n_features)
        # y shape: (samples, horizon)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_sequences(df, feature_cols, target_col, lookback, horizon):
    """
    Build sliding window sequences from a dataframe.
    X: multi-feature lookback window
    y: target (speed) for next horizon frames
    """
    features = df[feature_cols].values.astype(np.float32)
    target   = df[target_col].values.astype(np.float32)

    X, y = [], []
    for i in range(len(features) - lookback - horizon):
        X.append(features[i : i + lookback])           # (lookback, n_features)
        y.append(target[i + lookback : i + lookback + horizon])  # (horizon,)
    return np.array(X), np.array(y)


# Load data
master = pd.read_csv("../data/master_congestion_final.csv")

# Normalize features globally — important for multi-feature input
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Fit scaler on all cyclical data
cyclical_data = master[master['source'].isin(CYCLICAL_SOURCES)].copy()
cyclical_data[FEATURE_COLS] = scaler_X.fit_transform(
    cyclical_data[FEATURE_COLS]
)
cyclical_data['avg_speed_scaled'] = scaler_y.fit_transform(
    cyclical_data[['avg_speed']]
)

# Build sequences per location
all_X, all_y = [], []
for src in CYCLICAL_SOURCES:
    subset = cyclical_data[cyclical_data['source'] == src].sort_values('frame_num')
    X, y = make_sequences(subset, FEATURE_COLS,
                           'avg_speed_scaled', LOOKBACK, HORIZON)
    all_X.append(X)
    all_y.append(y)
    print(f"  {src:<35} sequences: {len(X)}")

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)
print(f"\nTotal:  X={all_X.shape}  y={all_y.shape}")

# Train / val / test split
n = len(all_X)
t1, t2 = int(n * 0.70), int(n * 0.85)

train_loader = DataLoader(
    MultiFeatureDataset(all_X[:t1], all_y[:t1]),
    batch_size=BATCH, shuffle=True
)
val_loader = DataLoader(
    MultiFeatureDataset(all_X[t1:t2], all_y[t1:t2]),
    batch_size=BATCH, shuffle=False
)
test_loader = DataLoader(
    MultiFeatureDataset(all_X[t2:], all_y[t2:]),
    batch_size=BATCH, shuffle=False
)
print(f"Train batches: {len(train_loader)}  "
      f"Val: {len(val_loader)}  Test: {len(test_loader)}")

  B.Dongying Road2                    sequences: 13990
  E.People's Hall1                    sequences: 14604
  E.People's Hall2                    sequences: 8879
  D.Xing'an South Road1               sequences: 8784
  D.Xing'an South Road2               sequences: 11728
  G.Xinhua Square1                    sequences: 23584
  G.Xinhua Square2                    sequences: 19129

Total:  X=(100698, 100, 4)  y=(100698, 30)
Train batches: 2203  Val: 473  Test: 473


In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=4, factor=0.5
)
criterion = nn.MSELoss()

best_val   = float('inf')
patience_c = 0
history    = {'train': [], 'val': []}

print(f"\nTraining multi-feature Transformer for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            pred = model(X_batch.to(device))
            val_loss += criterion(pred, y_batch.to(device)).item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)
    history['train'].append(train_loss)
    history['val'].append(val_loss)

    print(f"Epoch {epoch:03d} | train={train_loss:.5f} | val={val_loss:.5f}")

    # ── Early stopping ─────────────────────────────────────
    if val_loss < best_val:
        best_val   = val_loss
        patience_c = 0
        torch.save(model.state_dict(),
                   '../data/best_transformer_multifeature.pt')
    else:
        patience_c += 1
        if patience_c >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nBest val loss: {best_val:.5f}")
print(f"Saved: best_transformer_multifeature.pt")


Training multi-feature Transformer for 50 epochs...

Epoch 001 | train=0.04166 | val=0.03043
Epoch 002 | train=0.02960 | val=0.02798
Epoch 003 | train=0.02862 | val=0.02818
Epoch 004 | train=0.02763 | val=0.03872
Epoch 005 | train=0.02732 | val=0.02682
Epoch 006 | train=0.02678 | val=0.02968
Epoch 007 | train=0.02636 | val=0.02894
Epoch 008 | train=0.02598 | val=0.03238
Epoch 009 | train=0.02561 | val=0.03019
Epoch 010 | train=0.02537 | val=0.02826
Epoch 011 | train=0.02334 | val=0.03052
Epoch 012 | train=0.02304 | val=0.02981
Epoch 013 | train=0.02269 | val=0.02879
Epoch 014 | train=0.02234 | val=0.03238
Epoch 015 | train=0.02215 | val=0.02911

Early stopping at epoch 15

Best val loss: 0.02682
Saved: best_transformer_multifeature.pt


In [6]:
# Load best checkpoint
model.load_state_dict(
    torch.load('../data/best_transformer_multifeature.pt',
               map_location=device)
)
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        pred = model(X_batch.to(device)).cpu().numpy()
        all_preds.append(pred)
        all_true.append(y_batch.numpy())

preds = np.concatenate(all_preds)
trues = np.concatenate(all_true)

# Inverse transform to get back to km/h scale
preds_real = scaler_y.inverse_transform(preds)
trues_real = scaler_y.inverse_transform(trues)

mae  = np.mean(np.abs(preds_real - trues_real))
rmse = np.sqrt(np.mean((preds_real - trues_real) ** 2))
print(f"Test MAE  (km/h): {mae:.4f}")
print(f"Test RMSE (km/h): {rmse:.4f}")
print()
print("Compare with previous single-feature model:")
print("  Old MAE  (0-1 score scale): 0.0380")
print("  New MAE  (km/h scale): above — divide by ~10 to roughly compare")

Test MAE  (km/h): 0.1530
Test RMSE (km/h): 0.1982

Compare with previous single-feature model:
  Old MAE  (0-1 score scale): 0.0380
  New MAE  (km/h scale): above — divide by ~10 to roughly compare


In [7]:
# Reset model from scratch
model = MultiFeatureTransformer(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    nhead=N_HEADS,
    num_encoder_layers=N_LAYERS,
    dim_feedforward=D_FF,
    dropout=DROPOUT,
    lookback=LOOKBACK,
    horizon=HORIZON
).to(device)

# Lower LR, more patience
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=6, factor=0.5
)
criterion = nn.MSELoss()
EPOCHS    = 30
PATIENCE  = 15

best_val   = float('inf')
patience_c = 0
history    = {'train': [], 'val': []}

print("Retraining with lr=3e-4...\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            pred = model(X_batch.to(device))
            val_loss += criterion(pred, y_batch.to(device)).item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    print(f"Epoch {epoch:03d} | train={train_loss:.5f} | val={val_loss:.5f}")

    if val_loss < best_val:
        best_val   = val_loss
        patience_c = 0
        torch.save(model.state_dict(),
                   '../data/best_transformer_multifeature.pt')
    else:
        patience_c += 1
        if patience_c >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nBest val loss: {best_val:.5f}")

C:\Users\Jaya\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Retraining with lr=3e-4...

Epoch 001 | train=0.05102 | val=0.02835
Epoch 002 | train=0.02972 | val=0.02870
Epoch 003 | train=0.02755 | val=0.02797
Epoch 004 | train=0.02698 | val=0.02860
Epoch 005 | train=0.02593 | val=0.02715
Epoch 006 | train=0.02566 | val=0.02710
Epoch 007 | train=0.02507 | val=0.02791
Epoch 008 | train=0.02485 | val=0.02768
Epoch 009 | train=0.02456 | val=0.02933
Epoch 010 | train=0.02429 | val=0.02894
Epoch 011 | train=0.02426 | val=0.02899
Epoch 012 | train=0.02361 | val=0.02863
Epoch 013 | train=0.02376 | val=0.03110
Epoch 014 | train=0.02262 | val=0.02895
Epoch 015 | train=0.02234 | val=0.02856
Epoch 016 | train=0.02226 | val=0.02880
Epoch 017 | train=0.02221 | val=0.03078
Epoch 018 | train=0.02201 | val=0.03016
Epoch 019 | train=0.02189 | val=0.03225
Epoch 020 | train=0.02172 | val=0.03119
Epoch 021 | train=0.02099 | val=0.03053

Early stopping at epoch 21

Best val loss: 0.02710


In [8]:
import torch
import joblib
import json
import numpy as np

# ── Save the best transformer checkpoint ─────────────────
# Already saved during training as best_transformer_multifeature.pt
# Verify it exists
import os
path = '../data/best_transformer_multifeature.pt'
assert os.path.exists(path), f"Checkpoint not found at {path}"
print(f"✅ Transformer checkpoint: {path}")

# ── Save model config so you can reload it later ─────────
config = {
    'input_dim':          INPUT_DIM,
    'd_model':            D_MODEL,
    'nhead':              N_HEADS,
    'num_encoder_layers': N_LAYERS,
    'dim_feedforward':    D_FF,
    'dropout':            DROPOUT,
    'lookback':           LOOKBACK,
    'horizon':            HORIZON,
    'feature_cols':       FEATURE_COLS,
    'target_col':         TARGET_COL,
    'best_val_loss':      best_val,
    'epochs_trained':     epoch,
}
with open('../data/transformer_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✅ Transformer config: ../data/transformer_config.json")

# ── Save scalers ──────────────────────────────────────────
joblib.dump(scaler_X, '../data/scaler_X.pkl')
joblib.dump(scaler_y, '../data/scaler_y.pkl')
print("✅ Scalers: scaler_X.pkl, scaler_y.pkl")

# ── Save training history ─────────────────────────────────
np.save('../data/transformer_history.npy', history)
print("✅ Training history: transformer_history.npy")

print("\nAll transformer artifacts saved.")
print("\nFiles in ../data/:")
for f in sorted(os.listdir('../data/')):
    size = os.path.getsize(f'../data/{f}') / 1024
    print(f"  {f:<45} {size:.1f} KB")

✅ Transformer checkpoint: ../data/best_transformer_multifeature.pt
✅ Transformer config: ../data/transformer_config.json
✅ Scalers: scaler_X.pkl, scaler_y.pkl
✅ Training history: transformer_history.npy

All transformer artifacts saved.

Files in ../data/:
  A.Zhandong Road1.csv                          126693.6 KB
  A.Zhandong Road2.csv                          58021.0 KB
  B.Dongying Road1.csv                          148945.6 KB
  B.Dongying Road2.csv                          109604.0 KB
  C.Ulanqab East Street1.csv                    39537.3 KB
  C.Ulanqab East Street2.csv                    28484.7 KB
  C.Ulanqab East Street3.csv                    29079.4 KB
  C.Ulanqab East Street4.csv                    68655.1 KB
  D.Xing'an South Road1.csv                     37644.0 KB
  D.Xing'an South Road2.csv                     59391.3 KB
  E.People's Hall1.csv                          64823.5 KB
  E.People's Hall2.csv                          35600.1 KB
  F.South intersection1.csv     